<a href="https://colab.research.google.com/github/joaovitor73/Machine-Learning/blob/Master/avalicao_2_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### a) Contexto e Objetivo da Tarefa 1
Esta célula descreve o primeiro objetivo do notebook: avaliar 7 modelos classificadores de aprendizado de máquina usando validação cruzada para o dataset MNIST, baseando-se em notebooks de referência. Sugere modelos como Naive Bayes, MLP, SVM, Regressão Logística, SGD, KNN e XGBoost com seus hiperparâmetros padrão.

### Importação de Bibliotecas
Esta célula importa todas as bibliotecas necessárias para as tarefas de aprendizado de máquina, incluindo manipulação de dados (pandas, numpy), visualização (matplotlib), pré-processamento, seleção de modelos (sklearn), otimização de hiperparâmetros, e salvamento de modelos (joblib, pickle).

In [ ]:
#Importação de bibliotecas
from sklearn.datasets import fetch_openml
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import classification_report, accuracy_score, classification_report, roc_auc_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import SGDClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from scipy.stats import uniform, randint
from sklearn.model_selection import RandomizedSearchCV, train_test_split, GridSearchCV
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from statistics import mean
from sklearn.model_selection import KFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler, OrdinalEncoder
# Para incluir o SMOTE no pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# 5 Algoritmos de Classificação
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

import joblib
from sklearn.pipeline import Pipeline as SklearnPipeline
from sklearn.compose import ColumnTransformer

### Carregamento do Dataset MNIST
Esta célula carrega o dataset MNIST de reconhecimento de dígitos manuscritos (fetch_openml) e exibe suas chaves para inspeção.

In [ ]:
#importação da base mnist
mnist = fetch_openml('mnist_784', version=1, parser='auto')
mnist.keys()

dict_keys(['data', 'target', 'frame', 'categories', 'feature_names', 'target_names', 'DESCR', 'details', 'url'])

### Separação de Features e Target e Conversão de Tipo
Esta célula separa as features (X) e o target (y) do dataset MNIST. Converte o target 'y' para tipo inteiro e exibe as dimensões dos arrays resultantes.

In [ ]:
#Separando as features e o target
X, y = mnist.data, mnist.target
#
y = np.array(y, dtype=int)
print(X.shape)
print(y.shape)

(70000, 784)
(70000,)


### Redução de Dimensionalidade com PCA e Divisão Treino/Teste
Esta célula aplica a Análise de Componentes Principais (PCA) para reduzir a dimensionalidade dos dados (mantendo 95% da variância) e divide o dataset pré-processado em conjuntos de treino e teste.

In [ ]:
# Reduzindo a dimensionalidade da base
pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X)
X_train, X_test, y_train, y_test = X_pca[:60000], X_pca[60000:], y[:60000], y[60000:]

### Inicialização dos Modelos com Parâmetros Padrão
Esta célula define um dicionário contendo as instâncias dos 7 modelos classificadores sugeridos, todos com seus hiperparâmetros padrão, para a etapa de avaliação inicial.

In [ ]:
#Modelos com parametros default
models = {
    'Naive Bayes': GaussianNB(),
    'MLP': MLPClassifier(),
    'RandomForest': RandomForestClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(),
    'SGD': SGDClassifier(),
    'KNN': KNeighborsClassifier(),
    'XGBoost': XGBClassifier()
}


### Avaliação de Acurácia Média Global com Validação Cruzada
Esta célula realiza a avaliação de cada modelo utilizando validação cruzada (cross_val_predict) para obter as previsões e calcula a acurácia média global no conjunto de treino. Os resultados são armazenados e exibidos.

In [ ]:
accuracies = {}

for name, model in models.items():
    # Usando cross_val_predict para obter as previsões de validação cruzada
    predictions = cross_val_predict(model, X_train, y_train, cv=3)  # cv=3 para validação cruzada com 3 folds
    # Calculando a acurácia média
    accuracy = accuracy_score(y_train, predictions)
    accuracies[name] = accuracy

# Exibindo as acurácias médias dos modelos
for name, accuracy in accuracies.items():
    print(f'{name}: {accuracy:.4f}')

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

Naive Bayes: 0.8520
MLP: 0.9619
RandomForest: 0.9423
Logistic Regression: 0.9128
SGD: 0.8790
KNN: 0.9695
XGBoost: 0.9587


### b) Ranking dos Modelos pela Acurácia da Validação Cruzada
Esta célula aborda o segundo objetivo: ranquear os 7 modelos em ordem decrescente de acurácia global obtida na validação cruzada e identificar os três melhores (`top3`).

### Exibição do Ranking e Top 3 Modelos
Esta célula ordena as acurácias obtidas na validação cruzada e imprime o ranking dos modelos, destacando os 3 melhores classificadores ('KNN', 'MLP', 'XGBoost').

In [ ]:
sorted_cv = sorted(accuracies.items(), key=lambda x: x[1], reverse=True)
print("\nRanking (CV):")
for rank, (name, acc) in enumerate(sorted_cv, start=1):
    print(f"{rank}. {name}: {acc:.4f}")

top3 = [name for name, _ in sorted_cv[:3]]
print("\nTop 3 modelos:", top3)



Ranking (CV):
1. KNN: 0.9695
2. MLP: 0.9619
3. XGBoost: 0.9587
4. RandomForest: 0.9423
5. Logistic Regression: 0.9128
6. SGD: 0.8790
7. Naive Bayes: 0.8520

Top 3 modelos: ['KNN', 'MLP', 'XGBoost']


### c) Aplicação e Avaliação dos Modelos no Conjunto de Teste
Esta célula descreve o terceiro objetivo: aplicar os 7 modelos no conjunto de teste previamente separado para obter suas acurácias globais nesse conjunto.

### Treinamento e Avaliação dos Modelos no Conjunto de Teste
Esta célula treina cada um dos 7 modelos no `X_train` e `y_train` e, em seguida, calcula e exibe a acurácia de cada modelo no conjunto de `X_test` e `y_test`.

In [ ]:
accuracies_test = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred_test = model.predict(X_test)
    accuracies_test[name] = accuracy_score(y_test, y_pred_test)

print("\nAcurácias no conjunto de teste:")
for name, acc in accuracies_test.items():
    print(f"{name}: {acc:.4f}")


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Acurácias no conjunto de teste:
Naive Bayes: 0.8645
MLP: 0.9724
RandomForest: 0.9498
Logistic Regression: 0.9200
SGD: 0.8972
KNN: 0.9714
XGBoost: 0.9671


### d) Análise da Diferença entre Acurácias CV e Teste e Ranking
Esta célula detalha o quarto objetivo: calcular e ranquear os modelos com base nas menores diferenças absolutas entre suas acurácias de validação cruzada e de teste. A etapa também pede uma análise da coincidência entre o top-3 de validação cruzada e o top-3 de menor diferença.

In [ ]:
diffs = {name: abs(accuracies[name] - accuracies_test[name]) for name in models.keys()}

print("\nDiferenças |CV - Teste|:")
for name, diff in diffs.items():
    print(f"{name}: {diff:.4f}")

sorted_diff = sorted(diffs.items(), key=lambda x: x[1])
print("\nTop 3 com menores diferenças CV-teste:")
for rank, (name, diff) in enumerate(sorted_diff[:3], start=1):
    print(f"{rank}. {name}: {diff:.4f}")



Diferenças |CV - Teste|:
Naive Bayes: 0.0125
MLP: 0.0105
RandomForest: 0.0075
Logistic Regression: 0.0072
SGD: 0.0181
KNN: 0.0019
XGBoost: 0.0084

Top 3 com menores diferenças CV-teste:
1. KNN: 0.0019
2. Logistic Regression: 0.0072
3. RandomForest: 0.0075


e) realize grid search em cada modelo dos 3 acima (ou randomized search com
N combinações. Defina N igual para todos os modelos, mesmas sementes e
mesmo número de folds). Pesquise em IA os principais hiperparâmetros e
faixas de valores/opções sugeridas para cada um dos 3 algoritmos acima

In [ ]:
N = 5
cv_folds = 3
random_state = 42

# --- KNN ---
knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

knn_param_dist = {
    'knn__n_neighbors': randint(1, 21),
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['euclidean', 'manhattan', 'minkowski'],
    'knn__p': [1, 2]
}


rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(random_state=random_state))
])

rf_param_dist = {
    'rf__n_estimators': randint(50, 300),
    'rf__max_depth': randint(2, 20),
    'rf__min_samples_split': randint(2, 10),
    'rf__min_samples_leaf': randint(1, 10),
    'rf__max_features': ['sqrt', 'log2', None],
    'rf__bootstrap': [True, False]
}

# --- Logistic Regression ---
logreg_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=1000, random_state=random_state))
])

logreg_param_dist = {
    'logreg__C': uniform(0.01, 100),
    'logreg__penalty': ['l1', 'l2', 'elasticnet'],
    'logreg__solver': ['liblinear', 'saga', 'lbfgs'],
    'logreg__l1_ratio': [0.0, 0.1, 0.5, 0.9]
}


#=====================
def run_random_search(pipeline, param_dist, X_train, y_train, model_name):
    search = RandomizedSearchCV(
        pipeline,
        param_distributions=param_dist,
        n_iter=N,
        cv=cv_folds,
        random_state=random_state,
        n_jobs=-1,
        verbose=1
    )
    search.fit(X_train, y_train)
    print(f"\n--- Melhor modelo {model_name} ---")
    print(search.best_params_)
    print(f"Melhor score CV: {search.best_score_:.4f}")
    return search

# =====================
knn_search = run_random_search(knn_pipeline, knn_param_dist, X_train, y_train, "KNN")
rf_search = run_random_search(rf_pipeline, rf_param_dist, X_train, y_train, "Random Forest")
logreg_search = run_random_search(logreg_pipeline, logreg_param_dist, X_train, y_train, "Logistic Regression")

# =====================
# 5. Avaliar no conjunto de teste
# =====================
for model_name, model_search in zip(
    ["KNN", "Random Forest" "Logistic Regression"],
    [knn_search, rf_search, logreg_search]
):
    test_score = model_search.score(X_test, y_test)
    print(f"{model_name} acurácia no teste: {test_score:.4f}")

Fitting 3 folds for each of 5 candidates, totalling 15 fits

--- Melhor modelo KNN ---
{'knn__metric': 'minkowski', 'knn__n_neighbors': 2, 'knn__p': 2, 'knn__weights': 'distance'}
Melhor score CV: 0.9051
Fitting 3 folds for each of 5 candidates, totalling 15 fits

--- Melhor modelo Random Forest ---
{'rf__bootstrap': False, 'rf__max_depth': 13, 'rf__max_features': 'sqrt', 'rf__min_samples_leaf': 1, 'rf__min_samples_split': 4, 'rf__n_estimators': 108}
Melhor score CV: 0.9358
Fitting 3 folds for each of 5 candidates, totalling 15 fits


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
6 fits failed out of a total of 15.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
3 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py", line 662, in fit
    self._final_estimator.fit(Xt, y, **last


--- Melhor modelo Logistic Regression ---
{'logreg__C': np.float64(60.121501174320876), 'logreg__l1_ratio': 0.9, 'logreg__penalty': 'elasticnet', 'logreg__solver': 'saga'}
Melhor score CV: 0.9179
KNN acurácia no teste: 0.9103
Random ForestLogistic Regression acurácia no teste: 0.9388


f) com o best_model de cada grid search acima, aplique em um novo conjunto
de teste (não pode ser o do item d). Gere um novo split para ter um novo
conjunto de teste (ou seja, com outra semente)


e completo/curso/discplina/professor/data (

In [ ]:
# Novo seed e split
new_seed = 999
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_pca, y, test_size=0.2, stratify=y, random_state=new_seed
)

# Modelos com os melhores hiperparâmetros já encontrados
models = {
    'RandomForest': RandomForestClassifier(
        bootstrap=False,
        max_depth=13,
        max_features='sqrt',
        min_samples_leaf=1,
        min_samples_split=4,
        n_estimators=108,
        random_state=new_seed
    ),
    'Logistic Regression': LogisticRegression(
        C=60.121501174320876,
        l1_ratio=0.9,
        penalty='elasticnet',
        solver='saga',
        max_iter=5000,
        random_state=new_seed
    ),
    'KNN': KNeighborsClassifier(
        metric='minkowski',
        n_neighbors=2,
        p=2,
        weights='distance'
    ),
}

# Avaliar cada modelo no novo conjunto de teste
new_test_results = []

for model_name, model in models.items():
    model.fit(X_train2, y_train2)
    preds_new = model.predict(X_test2)
    acc_new = accuracy_score(y_test2, preds_new)
    new_test_results.append({
        "model": model_name,
        "new_test_accuracy": acc_new
    })

# Exibir resultados
for res in new_test_results:
    print(f"{res['model']}: {res['new_test_accuracy']:.4f}")

RandomForest: 0.9399
Logistic Regression: 0.9167
KNN: 0.9748


g) veja o melhor modelo e exporte o pickle (ou joblib) para consumo no
STREAMLIT, agora adaptado para ler qualquer dígito de 0 a 9.


In [ ]:
#exportar o RandomFlorest em pickle
import pickle

# Identificar o melhor modelo
best_result = max(new_test_results, key=lambda x: x['new_test_accuracy'])
best_model_name = best_result['model']
best_model = models[best_model_name]

print("\nMelhor modelo:", best_model_name)
print(f"Acurácia no novo teste: {best_result['new_test_accuracy']:.4f}")

# Exportar o melhor modelo em pickle junto com PCA
filename = f"best_model_{best_model_name.replace(' ', '_')}_PCA.pkl"
with open(filename, 'wb') as f:
    pickle.dump({'model': best_model, 'pca': pca}, f)

print(f"✅ Modelo e PCA salvos como: {filename}")


Melhor modelo: KNN
Acurácia no novo teste: 0.9748
✅ Modelo e PCA salvos como: best_model_KNN_PCA.pkl


============================================================

2) Classificação Binária de Empréstimos:
Com base no dataset de empréstimos (loan.csv) disponível em:
https://github.com/josenalde/machinelearning/blob/main/src/dataset/loan.csv,
desenvolver modelo de aprendizagem de máquina para classificação binária de
liberação de empréstimo. Sugere-se as etapas seguintes, mas não restritas a estas:
a) separe os conjuntos treino e teste

In [ ]:
#lendo a base de dados
df = pd.read_csv('loan.csv')
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [ ]:
cols_to_remove = ['Loan_ID', 'CoapplicantIncome', 'Loan_Amount_Term', 'Credit_History', 'Property_Area']
df = df.drop(columns=cols_to_remove)

In [ ]:
df.head()

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,LoanAmount,Loan_Status
0,Male,No,0,Graduate,No,5849,NaN,Y
1,Male,Yes,1,Graduate,No,4583,128.0,N
2,Male,Yes,0,Graduate,Yes,3000,66.0,Y
3,Male,Yes,0,Not Graduate,No,2583,120.0,Y
4,Male,No,0,Graduate,No,6000,141.0,Y


In [ ]:
df.isnull().sum()

,0
Gender,13
Married,3
Dependents,15
Education,0
Self_Employed,32
ApplicantIncome,0
LoanAmount,22
Loan_Status,0


In [ ]:
#Tratar 'dependents' 3+ -> 3 (Pré-etapa ANTES do pipeline)
df['Dependents'] = df['Dependents'].replace('3+', '3')

In [ ]:
y = LabelEncoder().fit_transform(df['Loan_Status'])
X = df.drop('Loan_Status', axis=1)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:
#mostrando a divisão
print(f"Tamanho do Treino: {X_train.shape}")
print(f"Tamanho do Teste: {X_test.shape}")
print(f"Tamanho do Treino: {y_train.shape}")
print(f"Tamanho do Teste: {y_test.shape}")

Tamanho do Treino: (429, 7)
Tamanho do Teste: (185, 7)
Tamanho do Treino: (429,)
Tamanho do Teste: (185,)


In [ ]:
X_train.head()

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,LoanAmount
197,Female,No,0,Not Graduate,No,1907,120.0
175,Male,Yes,0,Graduate,No,3497,116.0
526,Male,Yes,0,Graduate,No,3775,110.0
149,Male,Yes,0,Graduate,No,4860,125.0
507,NaN,No,0,Graduate,No,3583,96.0


In [ ]:
# Colunas numéricas (c) e (g)
numeric_features = ['ApplicantIncome', 'LoanAmount']
numeric_transformer = SklearnPipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Colunas categóricas (c) e (e)
# 'dependents' agora é ordinal (0, 1, 2, 3) e será tratada aqui
categorical_features = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed']
categorical_transformer = SklearnPipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # Preenche NaNs com moda
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)) # Converte para números
])

In [ ]:
# Combinar os transformadores em um único pré-processador
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough' # Mantém colunas não especificadas (se houver)
)

In [ ]:
param_grids = {
    'LogisticRegression': {
        'model__C': [0.1, 1.0, 10],
        'model__solver': ['liblinear']
    },
    'RandomForest': {
        'model__n_estimators': [100, 200],
        'model__max_depth': [5, 10, None]
    },
    'GradientBoosting': {
        'model__n_estimators': [100, 200],
        'model__learning_rate': [0.05, 0.1]
    },
    'SVC': {
        'model__C': [0.1, 1.0],
        'model__gamma': ['scale', 'auto'],
        'model__probability': [True] # Necessário para roc_auc
    },
    'KNeighbors': {
        'model__n_neighbors': [3, 5, 7],
        'model__weights': ['uniform', 'distance']
    }
}

classifiers = {
    'LogisticRegression': LogisticRegression(random_state=42, max_iter=1000),
    'RandomForest': RandomForestClassifier(random_state=42),
    'GradientBoosting': GradientBoostingClassifier(random_state=42),
    'SVC': SVC(random_state=42),
    'KNeighbors': KNeighborsClassifier()
}

In [ ]:
results = {}
best_model_pipeline = None
best_auc = -1
best_model_name = ""

for name in classifiers.keys():
    print(f"\n=======================================================")
    print(f"Treinando: {name}")
    print(f"=======================================================")

    # Criar o pipeline completo: Preprocessing -> SMOTE -> Model
    # Usamos o ImbPipeline para que o SMOTE seja aplicado
    # APENAS nos dados de treino de cada fold da validação cruzada
    pipeline = ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('smote', SMOTE(random_state=42)), # (b) SMOTE
        ('model', classifiers[name])
    ])

    # Configurar o GridSearchCV
    # Usamos scoring='roc_auc' para otimizar a métrica que você pediu
    grid_search = GridSearchCV(
        pipeline,
        param_grids[name],
        cv=5, # 5-fold cross-validation
        scoring='roc_auc',
        n_jobs=-1, # Usar todos os processadores
        verbose=1
    )

    # Treinar o Grid Search
    grid_search.fit(X_train, y_train)

    # Avaliar no conjunto de TESTE
    print(f"\nAvaliando {name} no conjunto de Teste...")
    best_pipe_from_grid = grid_search.best_estimator_
    y_pred = best_pipe_from_grid.predict(X_test)

    # Para AUC, precisamos das probabilidades da classe "1"
    y_proba = best_pipe_from_grid.predict_proba(X_test)[:, 1]

    # Calcular Métricas
    auc = roc_auc_score(y_test, y_proba)
    report = classification_report(y_test, y_pred, target_names=['Negado (0)', 'Aprovado (1)'])

    print(f"Melhores Hiperparâmetros: {grid_search.best_params_}")
    print(f"\nRelatório de Classificação (Teste):\n{report}")
    print(f"AUC (Teste): {auc:.4f}")
    print(f"Acurácia (Teste): {accuracy_score(y_test, y_pred):.4f}")

    results[name] = {'auc': auc, 'report': report, 'model': best_pipe_from_grid}

    # (i) Guardar o melhor modelo até agora
    if auc > best_auc:
        best_auc = auc
        best_model_name = name
        best_model_pipeline = best_pipe_from_grid

# -------------------------------------------------------------------------
# 5. (i) Escolha e Salvamento do Modelo Final
# -------------------------------------------------------------------------

print(f"\n=======================================================")
print(f"Modelo Final Escolhido (baseado na maior AUC): {best_model_name}")
print(f"Melhor AUC de Teste: {best_auc:.4f}")


Treinando: LogisticRegression
Fitting 5 folds for each of 3 candidates, totalling 15 fits

Avaliando LogisticRegression no conjunto de Teste...
Melhores Hiperparâmetros: {'model__C': 0.1, 'model__solver': 'liblinear'}

Relatório de Classificação (Teste):
              precision    recall  f1-score   support

  Negado (0)       0.36      0.53      0.43        58
Aprovado (1)       0.73      0.57      0.64       127

    accuracy                           0.56       185
   macro avg       0.54      0.55      0.53       185
weighted avg       0.61      0.56      0.57       185

AUC (Teste): 0.5159
Acurácia (Teste): 0.5568

Treinando: RandomForest
Fitting 5 folds for each of 6 candidates, totalling 30 fits

Avaliando RandomForest no conjunto de Teste...
Melhores Hiperparâmetros: {'model__max_depth': 10, 'model__n_estimators': 200}

Relatório de Classificação (Teste):
              precision    recall  f1-score   support

  Negado (0)       0.31      0.33      0.32        58
Aprovado (1)  

In [ ]:
import pickle

# Cria um dicionário com o modelo e o pré-processador
model_data = {
    "model": best_model_pipeline,
    "preprocessor": preprocessor
}

# Nome do arquivo final
filename = "final_loan_model.pkl"

# Salva tudo em um único arquivo pickle
with open(filename, "wb") as f:
    pickle.dump(model_data, f)

print(f"✅ Modelo e pré-processador salvos juntos em: {filename}")


✅ Modelo e pré-processador salvos juntos em: final_loan_model.pkl
